# Create and Train an Object Detection Model

This notebook builds a tiny COCO-format dataset, creates the detector from explicit backbone, neck, and head components, validates the project config, trains for one CPU epoch, then runs test and inference. Replace the synthetic dataset root with a prepared COCO folder when moving from the smoke example to real data.

Install runtime dependencies before running the training cells:

```bash
python -m pip install -e '.[cpu]'
```


In [ ]:
from __future__ import annotations

import importlib.util
import json
import shutil
import struct
import sys
import tempfile
import zlib
from dataclasses import asdict
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "simpledet" / "simpledet").is_dir():
            return candidate
    return start.resolve()


REPO_ROOT = find_repo_root(Path.cwd())
SOURCE_ROOT = REPO_ROOT / "simpledet"
if SOURCE_ROOT.is_dir() and str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

WORK_ROOT = Path(tempfile.gettempdir()) / "simpledet_object_detector_notebook"
DATASET_ROOT = WORK_ROOT / "dataset"
RUN_ROOT = WORK_ROOT / "runs" / "retinanet_demo"

print(f"repo root: {REPO_ROOT}")
print(f"work root: {WORK_ROOT}")


## Check the CPU Runtime

The native training path uses PyTorch, TorchVision, and Lightning. The check below fails early with the install command instead of failing halfway through training.


In [ ]:
missing = []
for module_name in ("torch", "torchvision"):
    if importlib.util.find_spec(module_name) is None:
        missing.append(module_name)

has_lightning = (
    importlib.util.find_spec("lightning") is not None
    or importlib.util.find_spec("pytorch_lightning") is not None
)
if not has_lightning:
    missing.append("lightning or pytorch_lightning")

if missing:
    raise RuntimeError(
        "Missing runtime dependency: "
        + ", ".join(missing)
        + ". Install with: python -m pip install -e '.[cpu]'"
    )

print("CPU runtime dependencies are available.")


## Build a Tiny COCO Dataset

SimpleDet expects COCO annotations plus an image directory. This cell writes three tiny splits so the notebook runs quickly on CPU and does not need external downloads.


In [ ]:
def write_png(path: Path, *, width: int = 32, height: int = 32) -> None:
    def chunk(kind: bytes, data: bytes) -> bytes:
        return (
            struct.pack(">I", len(data))
            + kind
            + data
            + struct.pack(">I", zlib.crc32(kind + data) & 0xFFFFFFFF)
        )

    rows = []
    for y in range(height):
        row = bytearray()
        for x in range(width):
            if 8 <= x < 24 and 8 <= y < 24:
                row.extend((220, 40, 40))
            else:
                row.extend((24, 32, 44))
        rows.append(b"\x00" + bytes(row))

    payload = (
        b"\x89PNG\r\n\x1a\n"
        + chunk(b"IHDR", struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0))
        + chunk(b"IDAT", zlib.compress(b"".join(rows)))
        + chunk(b"IEND", b"")
    )
    path.write_bytes(payload)


def write_coco_split(path: Path, *, split: str, image_id: int) -> None:
    file_name = f"{split}.png"
    payload = {
        "images": [
            {"id": image_id, "file_name": file_name, "width": 32, "height": 32}
        ],
        "annotations": [
            {
                "id": image_id,
                "image_id": image_id,
                "category_id": 1,
                "bbox": [8, 8, 16, 16],
                "area": 256,
                "iscrowd": 0,
            }
        ],
        "categories": [{"id": 1, "name": "object"}],
    }
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

(DATASET_ROOT / "images").mkdir(parents=True, exist_ok=True)
(DATASET_ROOT / "annotations").mkdir(parents=True, exist_ok=True)

for image_id, split in enumerate(("train", "val", "test"), start=1):
    write_png(DATASET_ROOT / "images" / f"{split}.png")
    write_coco_split(DATASET_ROOT / "annotations" / f"{split}.json", split=split, image_id=image_id)

sorted(str(path.relative_to(DATASET_ROOT)) for path in DATASET_ROOT.rglob("*"))


## Build and Connect Detector Components

The functional detector is assembled from four layers:

1. `build_backbone(...)` selects the feature extractor and its output stages.
2. `build_neck(...)` adapts the backbone feature maps into a feature pyramid.
3. `build_head(...)` turns each pyramid level into class and box predictions.
4. `build_detector(...)` glues those specs into one detector, and `compile_native_detector_plan(...)` shows the exact runtime connection contract.


In [ ]:
from simpledet.suite import (
    build_backbone,
    build_detector,
    build_head,
    build_neck,
    compile_native_detector_plan,
)

backbone = build_backbone(
    "resnet18",
    pretrained=False,
    in_channels=3,
    out_indices=(1, 2, 3, 4),
)
neck = build_neck("FPN", out_channels=64, num_outs=4)
head = build_head("RetinaHead", num_classes=1)

detector_spec = build_detector(
    "retinanet",
    backbone=backbone,
    neck=neck,
    head=head,
    num_classes=1,
    pretrained=False,
)
plan = compile_native_detector_plan(detector_spec)

connection_summary = {
    "detector": {"architecture": plan.architecture, "family": plan.family},
    "backbone_outputs": plan.encoder.params.get("feature_channels"),
    "neck": plan.neck.to_dict(),
    "head": plan.head.to_dict(),
}
print(json.dumps(connection_summary, indent=2, default=str))


## Materialize the Native Model

The compiled plan is not just a diagram. The native builder uses it to instantiate the backbone, connect its feature channels to the FPN, connect the FPN output width to the Retina head, and wrap everything in the detector module.


In [ ]:
from simpledet.native.modeling import build_detector as build_native_detector_module

model = build_native_detector_module(
    "retinanet",
    num_classes=1,
    in_channels=3,
    detector_spec=detector_spec,
)

print(type(model).__name__)
print(type(model.backbone).__name__, "->", type(model.neck).__name__, "->", type(model.head).__name__)


## Swap Specific Backbones, Necks, and Heads

Use the same glue pattern for other detector families. Dense detectors connect `backbone -> neck -> dense head`; ROI detectors connect `backbone -> neck -> RPN head -> ROI bbox/mask/grid heads` when the architecture requires proposals.


In [ ]:
def make_detector_recipe(
    architecture: str,
    *,
    backbone_name: str = "resnet18",
    neck_name: str = "FPN",
    head_name: str = "RetinaHead",
    num_classes: int = 1,
):
    recipe_backbone = build_backbone(
        backbone_name,
        pretrained=False,
        in_channels=3,
        out_indices=(1, 2, 3, 4),
    )
    recipe_neck = build_neck(neck_name, out_channels=64, num_outs=4)
    recipe_head = build_head(head_name, num_classes=num_classes)
    recipe_spec = build_detector(
        architecture,
        backbone=recipe_backbone,
        neck=recipe_neck,
        head=recipe_head,
        num_classes=num_classes,
        pretrained=False,
    )
    recipe_plan = compile_native_detector_plan(recipe_spec)
    recipe_model = build_native_detector_module(
        architecture,
        num_classes=num_classes,
        in_channels=3,
        detector_spec=recipe_spec,
    )
    return recipe_spec, recipe_plan, recipe_model


recipes = [
    ("retinanet", "resnet18", "FPN", "RetinaHead"),
    ("retinanet", "resnet50", "FPN", "RetinaHead"),
    ("retinanet", "convnext_tiny", "FPN", "RetinaHead"),
    ("fcos", "resnet18", "FPN", "FCOSHead"),
    ("faster_rcnn", "resnet18", "FPN", "Shared2FCBBoxHead"),
]

for architecture, backbone_name, neck_name, head_name in recipes:
    _, recipe_plan, recipe_model = make_detector_recipe(
        architecture,
        backbone_name=backbone_name,
        neck_name=neck_name,
        head_name=head_name,
    )
    route = {
        "architecture": recipe_plan.architecture,
        "family": recipe_plan.family,
        "model": type(recipe_model).__name__,
        "backbone": recipe_plan.encoder.type,
        "neck": recipe_plan.neck.type,
        "head": recipe_plan.head.type if recipe_plan.head else None,
        "rpn_head": recipe_plan.rpn_head.type if recipe_plan.rpn_head else None,
        "bbox_head": recipe_plan.bbox_head.type if recipe_plan.bbox_head else None,
    }
    print(json.dumps(route, indent=2, default=str))


## Compose the Training Project

The project config below trains the same explicit `detector_spec`. For real data, keep the same structure and point `dataset.root` at a prepared COCO folder.


In [ ]:
from simpledet import ProjectConfig, validate_project_config

project = ProjectConfig.from_mapping(
    {
        "stages": ["build", "train", "test", "infer"],
        "workdir": str(RUN_ROOT),
        "seed": 71,
        "detector_spec": asdict(detector_spec),
        "dataset": {
            "format": "coco",
            "root": str(DATASET_ROOT),
            "train": "annotations/train.json",
            "val": "annotations/val.json",
            "test": "annotations/test.json",
            "images_dir": "images",
            "classes": ["object"],
            "in_channels": 3,
        },
        "runtime": {
            "accelerator": "cpu",
            "devices": 1,
            "batch_size": 1,
            "max_epochs": 1,
            "num_workers": 0,
        },
        "optimizer": {"name": "SGD", "learning_rate": 0.001},
        "export": {"formats": ["json"]},
    }
)

validation = validate_project_config(project, strict=True)
print(json.dumps(validation, indent=2, default=str))


## Train, Test, and Run Inference

This runs a real Lightning training loop on CPU. The dataset is intentionally tiny, so the goal is workflow validation rather than useful accuracy.


In [ ]:
from simpledet import run_project

result = run_project(project, stages=("build", "train", "test", "infer"))

summary = {
    "stages": result["stages"],
    "manifest_path": result["manifest_path"],
    "checkpoint_path": result["train"].get("checkpoint_path"),
    "test_predictions": len(result["test"].get("predictions", [])),
    "infer_predictions": len(result["infer"].get("predictions", [])),
}
print(json.dumps(summary, indent=2, default=str))


## Inspect Outputs

The project runner writes a manifest, metrics, predictions, and a checkpoint under the work directory.


In [ ]:
output_files = sorted(path.relative_to(RUN_ROOT) for path in RUN_ROOT.rglob("*") if path.is_file())
print("Output files:")
for path in output_files:
    print(f"- {path}")

print("\nFirst inference prediction:")
print(json.dumps(result["infer"]["predictions"][0], indent=2, default=str))


## Use a Real COCO Dataset

Prepare COCO 2017 into the SimpleDet layout, then reuse the same project config with `dataset.root` set to the output folder:

```bash
PYTHONPATH=simpledet python scripts/prepare_coco_simpledet.py \
  --source-root /data/coco \
  --output-root ./data/coco \
  --train-limit 1000 \
  --val-limit 200 \
  --test-limit 200
```
